# 01 — Data Exploration

This notebook explores the **Tobi-Bueck Customer Support Tickets** dataset before any modeling.
The goal is to understand the structure of the data, the distribution of labels, and whether
tickets belonging to different priority levels or departments are linguistically distinguishable.

This matters for the core research question: if tickets across subgroups look the same,
we would not expect generation quality to differ. If they are systematically different,
there is a plausible mechanism for unequal response quality.

All figures and tables shown here are produced by `scripts/run_eda.py`.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, display
from src import config

## 1. Dataset overview

The dataset contains ~60K synthetic English-language customer support tickets.
After filtering to English only and applying the department scope (merging departments
with fewer than 1500 examples into **Other**), we retain **28,254 tickets**.

Each ticket has:
- `ticket`: the customer message
- `answer`: the agent response
- `priority`: one of `low`, `medium`, `high`
- `queue`: one of 5 named departments + `Other`

In [ ]:
summary = json.loads((config.TABLES / 'eda' / 'summary.json').read_text())
print(f"Total rows: {summary['rows']}")
print(f"Mean ticket tokens:  {summary['mean_ticket_tokens']:.1f}")
print(f"Mean answer tokens:  {summary['mean_answer_tokens']:.1f}")
print(f"Median ticket tokens: {summary['median_ticket_tokens']:.0f}")
print(f"Median answer tokens: {summary['median_answer_tokens']:.0f}")

## 2. Label distributions

**Priority** is moderately imbalanced: `low` is underrepresented (~20%) compared to `medium` and `high`.
This imbalance is relevant for classification (we use weighted loss) and for subgroup analysis
(smaller support means wider confidence intervals).

**Department** is heavily skewed: Technical Support alone covers ~29% of tickets,
while the smallest departments have only a few hundred examples.

In [ ]:
pri = pd.DataFrame(summary['priority_counts'].items(), columns=['priority', 'count'])
pri['%'] = (pri['count'] / pri['count'].sum() * 100).round(1)
pri

In [ ]:
display(Image(filename=str(config.FIGURES / 'eda' / 'priority_counts.png')))

In [ ]:
dep = pd.DataFrame(summary['department_counts'].items(), columns=['department', 'count'])
dep['%'] = (dep['count'] / dep['count'].sum() * 100).round(1)
dep.sort_values('count', ascending=False)

In [ ]:
display(Image(filename=str(config.FIGURES / 'eda' / 'department_counts.png')))

## 3. Priority × Department joint distribution

The heatmap below shows how tickets are distributed across the Cartesian product of priority and department.
A uniform distribution would suggest the two labels are independent.
Structural imbalances here would become confounders in subgroup generation analysis.

In [ ]:
display(Image(filename=str(config.FIGURES / 'eda' / 'priority_department_heatmap.png')))

## 4. Ticket and response length

Ticket and response lengths are remarkably uniform across priority levels (mean ~60 tokens for both).
This is an important observation: **length is not a confounder** for priority-based subgroup comparisons.
A model that performs worse on `low` priority tickets cannot attribute it to shorter or longer inputs.

In [ ]:
display(Image(filename=str(config.FIGURES / 'eda' / 'length_by_priority.png')))
display(Image(filename=str(config.FIGURES / 'eda' / 'ticket_length_by_department.png')))

## 5. Lexical separability (TF-IDF + SVD)

The scatter plot below projects tickets into 2D using TF-IDF followed by truncated SVD.
If subgroups form visually distinct clusters, the labels carry strong lexical signal.
Overlap between groups suggests that surface-level vocabulary alone is not sufficient
to distinguish them — which would make classification harder and generation more uniform.

This is a linear projection and loses most variance; the classification results in notebook 02
give a more reliable picture of separability.

In [ ]:
display(Image(filename=str(config.FIGURES / 'eda' / 'tfidf_svd_scatter.png')))

## Summary

Key findings from the data exploration:

- The dataset is **synthetic** — conclusions must be interpreted carefully.
- Priority is **moderately imbalanced** (`low` ~20%), department is **heavily skewed** (Technical Support ~29%).
- Ticket and response lengths are **uniform across priority** — length is not a confounder.
- The five smallest departments were merged into **Other** (threshold: < 1500 examples), leaving 6 department classes.
- Whether subgroups are linguistically separable is tested more rigorously in notebook 02 (classification).